In [ ]:
import joblib
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [2]:
df = pd.read_csv("cloudburst_dataset.csv")
df

,Cloud_Top_Height,Cloud_Base_Height,Lightning_Count,Optical_Thickness,Vertical_Development,Cloud_Movement,Rainfall,Humidity,Temperature,Pressure,Wind_Direction_Change,Updraft_Winds,cloud_burst,source_type
0,14102.922471,4478.254022,181.345045,81.458788,41.000000,3.100000,113.860000,98.456707,2.460000,831.976032,76.064499,46.290930,1,actual_based
1,16280.349921,3282.266452,135.691187,100.754718,41.000000,3.776932,145.754353,96.215862,5.351109,806.439829,111.885869,66.022711,1,actual_based
2,14610.191174,3540.533728,127.338520,79.712916,41.000000,3.100000,113.860000,100.342319,2.460000,832.275850,109.687108,53.216808,1,actual_based
3,14442.462510,5226.559823,123.329536,68.037289,41.000000,3.100000,136.020961,98.155915,5.775977,847.044026,66.602277,62.290350,1,actual_based
4,15792.328643,2259.156281,170.188910,65.791433,41.000000,3.100000,113.860000,100.163245,4.246368,847.358308,87.112221,71.000000,1,actual_based
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,12416.550137,4418.751713,43.000581,39.545973,19.097717,17.273247,158.989305,64.571139,33.088374,987.295359,117.378078,34.170900,0,normal_synthetic
9996,15214.536284,4427.930940,44.759519,30.847472,18.175465,26.116177,138.896527,63.000000,13.006152,985.756576,92.493156,21.382735,1,normal_synthetic
9997,12942.719214,1080.787694,20.691230,58.312846,17.557863,26.332929,137.858924,63.000000,30.256021,954.880262,51.188234,22.305768,1,normal_synthetic
9998,14245.382958,3919.624936,72.545314,45.259220,16.091781,25.250768,116.310489,67.982034,23.534713,992.333455,59.784394,25.059796,0,normal_synthetic


In [3]:
X = df.drop(columns=["cloud_burst", "source_type"])
y = df["cloud_burst"]
X

,Cloud_Top_Height,Cloud_Base_Height,Lightning_Count,Optical_Thickness,Vertical_Development,Cloud_Movement,Rainfall,Humidity,Temperature,Pressure,Wind_Direction_Change,Updraft_Winds
0,14102.922471,4478.254022,181.345045,81.458788,41.000000,3.100000,113.860000,98.456707,2.460000,831.976032,76.064499,46.290930
1,16280.349921,3282.266452,135.691187,100.754718,41.000000,3.776932,145.754353,96.215862,5.351109,806.439829,111.885869,66.022711
2,14610.191174,3540.533728,127.338520,79.712916,41.000000,3.100000,113.860000,100.342319,2.460000,832.275850,109.687108,53.216808
3,14442.462510,5226.559823,123.329536,68.037289,41.000000,3.100000,136.020961,98.155915,5.775977,847.044026,66.602277,62.290350
4,15792.328643,2259.156281,170.188910,65.791433,41.000000,3.100000,113.860000,100.163245,4.246368,847.358308,87.112221,71.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,12416.550137,4418.751713,43.000581,39.545973,19.097717,17.273247,158.989305,64.571139,33.088374,987.295359,117.378078,34.170900
9996,15214.536284,4427.930940,44.759519,30.847472,18.175465,26.116177,138.896527,63.000000,13.006152,985.756576,92.493156,21.382735
9997,12942.719214,1080.787694,20.691230,58.312846,17.557863,26.332929,137.858924,63.000000,30.256021,954.880262,51.188234,22.305768
9998,14245.382958,3919.624936,72.545314,45.259220,16.091781,25.250768,116.310489,67.982034,23.534713,992.333455,59.784394,25.059796


In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
xgb_clf = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="gpu_hist",
    random_state=42
)

In [6]:
param_grid = {
    "n_estimators": [100, 200, 300, 400, 500, 1000],
    "max_depth": [3, 5, 7, 9],
    "learning_rate": [0.001, 0.01, 0.05, 0.1, 0.2, 0.5],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

In [7]:
grid_search = GridSearchCV(
    estimator=xgb_clf,
    param_grid=param_grid,
    scoring="accuracy",
    cv=3,
    verbose=1,
    n_jobs=-1
)

In [8]:
grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)

Fitting 3 folds for each of 576 candidates, totalling 1728 fits


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [18:43:43] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  bst.update(dtrain, iteration=i, fobj=obj)


Best Parameters: {'colsample_bytree': 1.0, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 500, 'subsample': 0.8}
Best CV Score: 0.8576244642337937


In [9]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_pred))

Test Accuracy: 0.865


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:2676: UserWarning: [18:43:44] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  if len(data.shape) != 1 and self.num_features() != data.shape[1]:
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:729: UserWarning: [18:43:44] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [10]:
joblib.dump(grid_search.best_estimator_, "xgboost_model.pkl")

['xgboost_model.pkl']